# Perron: Cross-Architecture Scaling Surface & Generality Benchmark
### Protocol Reference: PREREG-SCALING-v1 (Held-Out SWE-bench Verified Stratum C)

**Author**: Sultan Haikal (GitHub: `@yvliet`)  
**Target**: Track 2 (Gemma 4 Developer Agent Paper Track)  
**Hardware**: Dual NVIDIA Tesla T4 GPUs (2x 15GB GDDR6)  
**License**: Apache-2.0

---

## 1. Executive Protocol & Hypotheses

Evaluates candidate discrimination (ROC-AUC) across a 5-tier 2025–2026 open-weight model gradient spanning:
1. **`google/gemma-4-e2b-it`** (2.0B Dense PLE, April 2026)
2. **`google/gemma-4-e4b-it`** (4.0B Dense PLE, April 2026)
3. **`microsoft/Phi-4-mini-instruct`** (3.8B Dense Reasoning SLM, Feb 2025)
4. **`deepseek-ai/DeepSeek-R1-Distill-Qwen-7B`** (7.0B Reasoning Distill, Jan 2025)
5. **`google/gemma-4-26b-a4b-it`** (3.8B active / 26B total MoE, April 2026)

Tests whether the **Lexical Abstraction Bottleneck** (AUC <= 0.6596) is scale-invariant zero-shot ($H_1$) and whether dynamic MoE routing provides representational efficiency over dense baselines ($H_2$).

In [ ]:
# Step 1: Hardware Environment, CUDA Telemetry & Secrets Setup
import os, sys, psutil, time, json, gc, re, shutil, traceback
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from pathlib import Path
import torch

# Attach Hugging Face token if configured in Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    token = secrets.get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = token
        print('[+] Attached HF_TOKEN from Kaggle Secrets.')
except Exception:
    pass

print(f'Python Version: {sys.version.split()[0]}')
print(f'PyTorch Version: {torch.__version__}')
cuda_ok = torch.cuda.is_available()
num_gpus = torch.cuda.device_count() if cuda_ok else 0
print(f'CUDA Available: {cuda_ok} (Count: {num_gpus})')

if cuda_ok:
    for i in range(num_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f'  [GPU {i}]: {props.name} | Total VRAM: {props.total_memory / (1024**3):.2f} GB')
else:
    print('Operating in CPU / Emulation mode.')


In [ ]:
# Step 2: Install dependencies if missing
!pip install -q bitsandbytes accelerate scipy transformers


In [ ]:
# Step 3: Locate Stratum C Evaluation Bundle
bundle_candidates = [
    Path('/kaggle/input/perron-gemma4-benchmark/stratum_c_eval_bundle.json'),
    Path('/kaggle/input/stratum_c_eval_bundle.json'),
    Path('data_release/stratum_c_eval_bundle.json'),
    Path('../data_release/stratum_c_eval_bundle.json'),
]

bundle_path = None
for p in bundle_candidates:
    if p.exists():
        bundle_path = p
        break

if bundle_path is None:
    base_in = Path('/kaggle/input')
    if base_in.exists():
        for p in base_in.rglob('stratum_c_eval_bundle.json'):
            if p.exists():
                bundle_path = p
                break

if bundle_path is None:
    raise FileNotFoundError('Could not find stratum_c_eval_bundle.json in /kaggle/input or local paths.')

print(f'[+] Using evaluation bundle: {bundle_path}')
with open(bundle_path, 'r', encoding='utf-8') as f:
    bundle_data = json.load(f)
tasks = bundle_data.get('tasks', [])
print(f'[+] Loaded {len(tasks)} tasks.')


In [ ]:
# Step 4: Model Checkpoint Resolver, Cache Purger & Scoring Primitives
import numpy as np

def resolve_model_path(model_id: str) -> str:
    if Path(model_id).exists() and (Path(model_id) / 'config.json').exists():
        return model_id
    base_in = Path('/kaggle/input')
    if base_in.exists():
        slug = model_id.split('/')[-1].lower()
        for p in base_in.rglob('config.json'):
            if slug in str(p.parent).lower():
                print(f'[+] Found local Kaggle mount for {model_id}: {p.parent}')
                return str(p.parent)
        if 'gemma' in slug:
            for variant in ['e2b', 'e4b', '26b', '31b', '12b']:
                if variant in slug:
                    for p in base_in.rglob('config.json'):
                        p_str = str(p.parent).lower()
                        if variant in p_str and 'gemma' in p_str:
                            print(f'[+] Resolved local Kaggle mount for {model_id} (key={variant}): {p.parent}')
                            return str(p.parent)
    return model_id

def purge_hf_cache():
    for cdir in [Path('/root/.cache/huggingface/hub'), Path(os.path.expanduser('~/.cache/huggingface/hub'))]:
        if cdir.exists():
            print(f'[+] Purging Hugging Face cache at {cdir} to reclaim scratch disk...')
            shutil.rmtree(cdir, ignore_errors=True)

PROMPT_P0 = (
    "You are an expert software engineer reviewing a repository to locate a reported defect.\n\n"
    "Problem Statement:\n{issue_text}\n\n"
    "Candidate Code Element:\n{candidate_breadcrumb}\n\n"
    "Does this candidate code element contain or directly cause the reported bug? Answer 'Yes' or 'No'."
)

def truncate_issue_text(text: str, max_chars: int = 3000) -> str:
    if not text: return ''
    clean = re.sub(r'\s+', ' ', text).strip()
    if len(clean) <= max_chars: return clean
    half = (max_chars - 60) // 2
    return clean[:half] + ' ... [TRUNCATED] ... ' + clean[-half:]

def format_candidate_breadcrumb(cand: dict) -> str:
    sym_name = cand.get('symbol_name') or cand.get('identifier') or 'unknown'
    fpath = cand.get('file_path') or cand.get('file') or 'unknown'
    lines = cand.get('lines') or [cand.get('line_start', 1), cand.get('line_end', 1)]
    snippet = cand.get('code') or cand.get('snippet') or f'# {sym_name}'
    return f'File: {fpath} (Lines {lines[0]}-{lines[1]})\nSymbol: {sym_name}\n```python\n{snippet}\n```'

def resolve_yes_no_token_ids(tok):
    y_ids, n_ids = set(), set()
    for w in ['Yes', ' Yes', 'YES', ' yes']:
        enc = tok.encode(w, add_special_tokens=False)
        if enc: y_ids.add(enc[-1])
    for w in ['No', ' No', 'NO', ' no']:
        enc = tok.encode(w, add_special_tokens=False)
        if enc: n_ids.add(enc[-1])
    return sorted(list(y_ids)), sorted(list(n_ids))

def forward_last_logits(m, inputs):
    try:
        res = m(**inputs, logits_to_keep=1, use_cache=False)
        return res.logits[:, -1, :]
    except TypeError:
        try:
            res = m(**inputs, use_cache=False)
            return res.logits[:, -1, :]
        except TypeError:
            res = m(**inputs)
            return res.logits[:, -1, :]


In [ ]:
# Step 5: Benchmark Execution Across 5 Modern Models (With Fail-Fast Circuit Breaker)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

PRIOR_RESULTS = [
    {
        'id': 'google/gemma-4-e2b-it',
        'name': 'Gemma 4 E2B',
        'params_b': 2.0,
        'active_params_b': 2.0,
        'archetype': 'PLE Dense Edge',
        'tier': 'pointwise',
        'live_model_executed': True,
        'device': 'Tesla T4',
        'gpu_vram_breakdown_mb': [6698.91455078125, 0.0],
        'peak_vram_mb': 6698.91455078125,
        'elapsed_seconds': 1797.9554815292358,
        'mean_seed_auc': 0.5,
        'mean_graph_auc': 0.8544116202940206,
        'n_tasks': 58,
        'listwise_anchor_metrics': {},
        'status': 'success'
    },
    {
        'id': 'google/gemma-4-e4b-it',
        'name': 'Gemma 4 E4B',
        'params_b': 4.0,
        'active_params_b': 4.0,
        'archetype': 'PLE Dense Edge',
        'tier': 'hybrid_anchor',
        'live_model_executed': True,
        'device': 'Tesla T4',
        'gpu_vram_breakdown_mb': [9648.3759765625, 0.0],
        'peak_vram_mb': 9648.3759765625,
        'elapsed_seconds': 6250.544926166534,
        'mean_seed_auc': 0.5,
        'mean_graph_auc': 0.927415136504036,
        'n_tasks': 58,
        'listwise_anchor_metrics': {
            'acc_top1': 0.015267175572519083,
            'acc_top5': 0.007633587786259542,
            'mrr': 0.015267175572519083,
            'hits_top1': 2,
            'hits_top5': 1
        },
        'status': 'success'
    },
    {
        'id': 'microsoft/Phi-4-mini-instruct',
        'name': 'Phi-4-mini',
        'params_b': 3.8,
        'active_params_b': 3.8,
        'archetype': 'High-Density Synthetic Reasoning',
        'tier': 'pointwise',
        'live_model_executed': True,
        'device': 'Tesla T4',
        'gpu_vram_breakdown_mb': [4135.91162109375, 0.0],
        'peak_vram_mb': 4135.91162109375,
        'elapsed_seconds': 2854.374233484268,
        'mean_seed_auc': 0.5,
        'mean_graph_auc': 0.7835563422056073,
        'n_tasks': 58,
        'listwise_anchor_metrics': {},
        'status': 'success'
    },
    {
        'id': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B',
        'name': 'DeepSeek R1 Distill 7B',
        'params_b': 7.0,
        'active_params_b': 7.0,
        'archetype': 'RL Reasoning Distill',
        'tier': 'pointwise',
        'live_model_executed': True,
        'device': 'Tesla T4',
        'gpu_vram_breakdown_mb': [6594.6181640625, 0.0],
        'peak_vram_mb': 6594.6181640625,
        'elapsed_seconds': 5230.614417314529,
        'mean_seed_auc': 0.5,
        'mean_graph_auc': 0.6488732627756683,
        'n_tasks': 58,
        'listwise_anchor_metrics': {},
        'status': 'success'
    }
]

MODELS_CONFIG = [
    {'id': 'google/gemma-4-26b-a4b-it', 'name': 'Gemma 4 26B-A4B', 'params_b': 26.0, 'active_params_b': 3.8, 'archetype': 'Dynamic Sparse MoE', 'tier': 'hybrid_anchor'},
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
) if torch.cuda.is_available() else None

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
all_model_results = list(PRIOR_RESULTS)

for m_spec in MODELS_CONFIG:
    m_id = m_spec['id']
    target_path = resolve_model_path(m_id)
    is_local = Path(target_path).exists()

    print('=' * 80)
    print(f'Evaluating Model: {m_id} (Target Path: {target_path}, Local: {is_local})')
    print('=' * 80)
    t_start = time.time()
    model, tok = None, None
    out, last_logits, inp, gen_toks = None, None, None, None

    try:
        # Reset GPU telemetry counters and reclaim residual cache
        if torch.cuda.is_available():
            gc.collect()
            torch.cuda.empty_cache()
            for i in range(torch.cuda.device_count()):
                torch.cuda.reset_peak_memory_stats(i)

        load_kwargs = {'trust_remote_code': False}
        if not is_local and hf_token:
            load_kwargs['token'] = hf_token

        tok = AutoTokenizer.from_pretrained(target_path, **load_kwargs)
        if tok.pad_token is None: tok.pad_token = tok.eos_token
        tok.padding_side = 'left'

        if torch.cuda.is_available():
            num_gpus = torch.cuda.device_count()
            target_dmap = 'auto' if (num_gpus > 1 and m_spec['params_b'] >= 20.0) else 'cuda:0'
            model_kwargs = {
                'quantization_config': bnb_config,
                'device_map': target_dmap,
                'trust_remote_code': False,
                'torch_dtype': torch.float16,
            }
            if target_dmap == 'auto':
                model_kwargs['max_memory'] = {
                    0: '14.0GiB',
                    1: '14.0GiB',
                }
            if not is_local and hf_token:
                model_kwargs['token'] = hf_token
            try:
                model = AutoModelForCausalLM.from_pretrained(target_path, **model_kwargs)
            except Exception as load_err:
                print(f'[!] Initial load with max_memory failed: {load_err}. Retrying with auto device map without explicit max_memory...')
                model_kwargs.pop('max_memory', None)
                model = AutoModelForCausalLM.from_pretrained(target_path, **model_kwargs)
            emb = model.get_input_embeddings() if hasattr(model, 'get_input_embeddings') else None
            input_device = emb.weight.device if (emb is not None and hasattr(emb, 'weight')) else next(model.parameters()).device
            print(f'[+] Model loaded on device_map="{target_dmap}" | Input device: {input_device}')
        else:
            model = None
            input_device = torch.device('cpu')

        y_ids, n_ids = resolve_yes_no_token_ids(tok)
        print(f'Classification token IDs: Yes={y_ids}, No={n_ids}')
        assert len(y_ids) > 0 and len(n_ids) > 0, f'Failed to resolve classification token IDs for {m_id}'

        # --- Pre-Flight Smoke Test Probe (Task 0 Verification) ---
        print(f'[Smoke Test] Verifying model {m_id} on synthetic candidate probe...')
        smoke_text = 'Problem Statement: Verification.\n\nCandidate: def test(): pass\n\nDoes this cause the bug? Answer Yes or No.'
        smoke_inp = tok(smoke_text, return_tensors='pt', max_length=512, truncation=True)
        if torch.cuda.is_available() and model is not None:
            smoke_inp = smoke_inp.to(input_device)
            with torch.inference_mode():
                s_logits = forward_last_logits(model, smoke_inp)
                s_y = torch.tensor(y_ids, dtype=torch.long, device=s_logits.device)
                s_n = torch.tensor(n_ids, dtype=torch.long, device=s_logits.device)
                s_score = (torch.logsumexp(s_logits[:, s_y], dim=-1) - torch.logsumexp(s_logits[:, s_n], dim=-1)).cpu().item()
            print(f'[Smoke Test] Forward pass verified! (Sample log-odds: {s_score:.4f})')

            # Stress test long sequence to certify zero OOM on real SWE-bench context
            long_text = smoke_text + '\n' + ('def probe_context_stress_block():\n    x = 1\n    return x\n' * 120)
            long_inp = tok(long_text, return_tensors='pt', max_length=2048, truncation=True).to(input_device)
            with torch.inference_mode():
                _ = forward_last_logits(model, long_inp)
            print(f'[Smoke Test] Long context probe verified ({long_inp.input_ids.shape[1]} tokens)!')

            if m_spec['tier'] == 'hybrid_anchor':
                with torch.inference_mode():
                    _ = model.generate(**smoke_inp, max_new_tokens=5, temperature=0.0, do_sample=False, pad_token_id=tok.eos_token_id)
                print('[Smoke Test] Autoregressive generation verified!')
        print(f'[Smoke Test] PASSED - Model {m_id} is certified operational.')

        # --- Pointwise Candidate Discrimination Loop ---
        task_aucs = []
        for t_idx, t in enumerate(tasks):
            issue = truncate_issue_text(t.get('problem_statement', ''))
            cands = t.get('candidates', [])
            gold_ids = set(t.get('gold_symbol_ids', [])) or set(t.get('gold_symbol_names', []))
            if not cands or not gold_ids: continue

            prompts = []
            for c in cands:
                bc = format_candidate_breadcrumb(c)
                raw = PROMPT_P0.format(issue_text=issue, candidate_breadcrumb=bc)
                if hasattr(tok, 'apply_chat_template'):
                    try: fmt = tok.apply_chat_template([{'role': 'user', 'content': raw}], add_generation_prompt=True, tokenize=False)
                    except Exception: fmt = raw
                else: fmt = raw
                prompts.append(fmt)

            scores = []
            if model is not None:
                for p_text in prompts:
                    inp = tok(p_text, return_tensors='pt', truncation=True, max_length=2048).to(input_device)
                    with torch.inference_mode():
                        last_logits = forward_last_logits(model, inp)
                        y_tensor = torch.tensor(y_ids, dtype=torch.long, device=last_logits.device)
                        n_tensor = torch.tensor(n_ids, dtype=torch.long, device=last_logits.device)
                        y_log = torch.logsumexp(last_logits[:, y_tensor], dim=-1)
                        n_log = torch.logsumexp(last_logits[:, n_tensor], dim=-1)
                        scores.append(float((y_log - n_log).cpu().item()))
            else:
                scores = [float(np.random.normal(0, 1)) for _ in prompts]

            g_scores = [s for c, s in zip(cands, scores) if (c.get('symbol_id') or c.get('identifier')) in gold_ids]
            d_scores = [s for c, s in zip(cands, scores) if (c.get('symbol_id') or c.get('identifier')) not in gold_ids]
            if g_scores and d_scores:
                wins = sum(1.0 if g > d else 0.5 if g == d else 0.0 for g in g_scores for d in d_scores)
                task_aucs.append({
                    'instance_id': t['instance_id'],
                    'auc': wins / (len(g_scores) * len(d_scores)),
                    'is_seed': any(c.get('is_seed', False) for c in cands if (c.get('symbol_id') or c.get('identifier')) in gold_ids)
                })

            if (t_idx + 1) % 25 == 0 or (t_idx + 1) == len(tasks):
                current_mean_auc = np.mean([r['auc'] for r in task_aucs]) if task_aucs else 0.5
                print(f'[{m_spec["name"]}] Evaluated task {t_idx+1}/{len(tasks)} (running mean AUC: {current_mean_auc:.4f})')
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()


        # --- Listwise Anchor Reasoning Loop ---
        listwise_metrics = {}
        if m_spec['tier'] == 'hybrid_anchor' and model is not None:
            print(f'Executing Listwise Anchor Reasoning for {m_id}...')
            hits_top1 = 0
            hits_top5 = 0
            reciprocal_ranks = []
            for t_idx, t in enumerate(tasks):
                gold_ids = set(t.get('gold_symbol_ids', [])) or set(t.get('gold_symbol_names', []))
                cands = t.get('candidates', [])[:25]
                if not cands or not gold_ids: continue
                cand_list_text = '\n'.join([f'[{i+1}] {c.get("symbol_name") or c.get("identifier")} (File: {c.get("file_path")})' for i, c in enumerate(cands)])
                prompt_text = (
                    f'You are debugging the following issue:\n{truncate_issue_text(t.get("problem_statement", ""))}\n\n'
                    f'Candidate Functions:\n{cand_list_text}\n\n'
                    'Think carefully inside <thought>...</thought> tags, then state the single best candidate index: [INDEX].'
                )
                if hasattr(tok, 'apply_chat_template'):
                    try: fmt_p = tok.apply_chat_template([{'role': 'user', 'content': prompt_text}], add_generation_prompt=True, tokenize=False)
                    except Exception: fmt_p = prompt_text
                else: fmt_p = prompt_text

                inp = tok(fmt_p, return_tensors='pt', max_length=3000, truncation=True).to(input_device)
                with torch.inference_mode():
                    gen_toks = model.generate(**inp, max_new_tokens=180, temperature=0.0, do_sample=False, pad_token_id=tok.eos_token_id)
                out_str = tok.decode(gen_toks[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
                pred_idx = 1
                m_match = re.search(r'(?:candidate\s*:?\s*|index\s*:?\s*|\[)\s*(\d+)', out_str, re.IGNORECASE)
                if m_match: pred_idx = int(m_match.group(1))
                pred_cid = None
                if 1 <= pred_idx <= len(cands):
                    pred_cid = cands[pred_idx - 1].get('symbol_id') or cands[pred_idx - 1].get('identifier')
                is_hit = pred_cid in gold_ids
                if is_hit:
                    hits_top1 += 1
                    reciprocal_ranks.append(1.0)
                else:
                    reciprocal_ranks.append(0.0)
                hits_top5 += int(pred_idx <= 5 and is_hit)

                if (t_idx + 1) % 25 == 0 or (t_idx + 1) == len(tasks):
                    print(f'[{m_spec["name"]} Listwise] Task {t_idx+1}/{len(tasks)} (top1 hits: {hits_top1}, top5 hits: {hits_top5})')
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

            listwise_metrics = {
                'acc_top1': hits_top1 / len(tasks) if tasks else 0.0,
                'acc_top5': hits_top5 / len(tasks) if tasks else 0.0,
                'mrr': float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0,
                'hits_top1': hits_top1,
                'hits_top5': hits_top5,
            }
            print(f'Listwise Anchor Result: Acc@1={listwise_metrics["acc_top1"]:.4f} | Acc@5={listwise_metrics["acc_top5"]:.4f} | MRR={listwise_metrics["mrr"]:.4f}')

        seed_aucs = [r['auc'] for r in task_aucs if r['is_seed']]
        graph_aucs = [r['auc'] for r in task_aucs if not r['is_seed']]
        gpu_vrams = [float(torch.cuda.max_memory_allocated(i) / (1024**2)) for i in range(num_gpus)] if torch.cuda.is_available() else [0.0]
        peak_vram = sum(gpu_vrams)
        elapsed = time.time() - t_start

        m_res = {
            **m_spec,
            'live_model_executed': torch.cuda.is_available(),
            'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
            'gpu_vram_breakdown_mb': gpu_vrams,
            'peak_vram_mb': float(peak_vram),
            'elapsed_seconds': float(elapsed),
            'mean_seed_auc': float(np.mean(seed_aucs)) if seed_aucs else 0.5,
            'mean_graph_auc': float(np.mean(graph_aucs)) if graph_aucs else 0.5,
            'n_tasks': len(task_aucs),
            'listwise_anchor_metrics': listwise_metrics,
            'status': 'success',
        }
        print(f'Result: Seed AUC={m_res["mean_seed_auc"]:.4f} | Graph AUC={m_res["mean_graph_auc"]:.4f} | Peak VRAM={peak_vram:.1f} MB (GPUs: {gpu_vrams})')
        all_model_results.append(m_res)

    except Exception as e:
        print(f'CRITICAL FAILURE evaluating {m_id}: {e}')
        traceback.print_exc()
        # Dump partial checkpoint to working directory
        partial_payload = {'status': 'aborted_early', 'failed_model': m_id, 'error': str(e), 'partial_results': all_model_results}
        with open('/kaggle/working/partial_results.json', 'w', encoding='utf-8') as pf:
            json.dump(partial_payload, pf, indent=2)
        # Fail-fast circuit breaker: halt immediately to prevent GPU quota burn
        raise RuntimeError(f'Aborting benchmark execution early to prevent GPU quota burn on failed model: {m_id}') from e

    finally:
        model = None
        tok = None
        out = None
        last_logits = None
        inp = None
        gen_toks = None
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        if not is_local:
            purge_hf_cache()


In [ ]:
# Step 6: Statistical Synthesis & Receipt Export
from scipy import stats
import datetime

valid = [m for m in all_model_results if m.get('status') == 'success']
tau_val, tau_pval = None, None
if len(valid) >= 3:
    caps = [m['params_b'] for m in valid]
    gaucs = [m['mean_graph_auc'] for m in valid]
    tau_val, tau_pval = stats.kendalltau(caps, gaucs)
    print(f'Hypothesis H1 (Monotonic Scaling): Kendall tau = {tau_val:.4f} (p = {tau_pval:.4e})')

out_payload = {
    'metadata': {
        'protocol_id': 'PREREG-SCALING-v1',
        'timestamp': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'n_models_evaluated': len(all_model_results),
        'n_models_succeeded': len(valid),
    },
    'hypothesis_tests': {
        'H1_monotonic_scaling': {
            'kendall_tau': float(tau_val) if tau_val is not None else None,
            'p_value': float(tau_pval) if tau_pval is not None else None,
        }
    },
    'models': all_model_results,
}

out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('results')
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / 'scaling_benchmark_results.json'

with open(out_file, 'w', encoding='utf-8') as f:
    json.dump(out_payload, f, indent=2)

print(f'[SUCCESS] Verified empirical receipts exported to: {out_file}')
